<a href="https://colab.research.google.com/github/shreyaghora/ML/blob/main/Code/ADASYN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
    make_scorer
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

from imblearn.over_sampling import SMOTE, ADASYN

# G-Mean is often imported from imbalanced-learn or calculated manually
from imblearn.metrics import geometric_mean_score

In [ ]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

In [ ]:
# Droping duration column to prevent leakage
df.drop('duration', axis=1, inplace=True) #drop method used for removing specified rows and column from a dataset

# For removing unknown values like 0, null, ?
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

In [ ]:
#Encoding in binary[0 & 1] values i.e. Final value or, Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# One-Hot Encoding: Convert ordinary data into numeric form [Only 1 column is (hot)1 at a time in each row for N categories]
df = pd.get_dummies(df, drop_first=True)
#Scaling: Step of preprocessing
scaler = StandardScaler()

In [ ]:
# Split Features(X:Input variables used for train the model and make prediction) & Target(y:Variable the model is intend to predict)

X = df.drop('y', axis=1)
y = df['y']

#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
#Adasyn(Sampling) Techniques
adasyn = ADASYN(random_state=42)
X_train_sm, y_train_sm = adasyn.fit_resample(X_train, y_train)

In [ ]:
#Models
models = {
    "Logistic Regression": LogisticRegression
     (
         max_iter=1000,
         class_weight='balanced',
         random_state=42,
         C=1.0
      ),
    "Decision Tree": DecisionTreeClassifier
     (
            max_depth=20,
            min_samples_split=40,
            min_samples_leaf=2,
            random_state=42

    ),
    "Random Forest": RandomForestClassifier
     (
        random_state=42,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
     ),
    "GBM": XGBClassifier
     (
         random_state=42,
         learning_rate=0.3,
         n_estimators=20
     ),
    "XGBoost": XGBClassifier
     (
         random_state=42,
         n_estimators=100,
         max_depth=3,
         learning_rate=0.3,
      ),
    "LightGBM": XGBClassifier
     (
         max_depth=[3,5],
         learning_rate=0.3,
         n_estimators=100
     ),
    "CatBoost": XGBClassifier
    (
        n_estimators=500,
        max_depth=3,
        learning_rate=0.1,
    ),
    "SVM": LinearSVC(random_state=42, C=1.0,tol=1e-3, max_iter=2000),
    "KNN": KNeighborsClassifier
     (
         n_neighbors=15,
         weights='distance',

     ),
    "MLP": MLPClassifier
     (
         random_state=42,
         hidden_layer_sizes=(64,32),
         activation='relu',
         solver='adam',
         max_iter=200,
         early_stopping=True
         ),
    "BaggingClassifier": BaggingClassifier
     (
         random_state=42,
         n_estimators=200,
         max_samples=0.8,
         max_features=0.8
         ),
    "StackingClassifier": StackingClassifier
     (
        cv=10,
        estimators=[
            ('lr', LogisticRegression(random_state=42)),
            ('rf', RandomForestClassifier(random_state=42)),
            ('dt', DecisionTreeClassifier(random_state=42)),
        ],
        n_jobs=-1,
        passthrough=True,
        verbose=1,
     )

  }

In [ ]:
#Hyperparameter of models for random search
params= {
    "Logistic Regression": {
                    'C':[1.0]
                    },
    "Decision Tree":{
            'max_depth': [20],
            'min_samples_split': [40],
            'min_samples_leaf': [2]
        },

    "Random Forest": {
            'max_depth': [None],
            'min_samples_split': [2],
            'min_samples_leaf': [1]
        },
    "GBM":{
            'n_estimators': [20],
            'learning_rate': [0.3]
    },
    "XGBoost":{
            'learning_rate': [0.3],
            'max_depth': [3],
            'n_estimators': [100]
    },
    "LightGBM":{
        'max_depth': [3],
        'learning_rate': [0.3],
        'n_estimators': [100]
    },
    "CatBoost":{
        'n_estimators': [500],
        'max_depth': [3],
        'learning_rate': [0.1]
    },
    "SVM": {
        'C':loguniform(1e-4,1e2),
        'loss':['hinge','squared_hinge'],
        'penalty':['l2'],
        'tol':loguniform(1e-4,1e-2)
    },
    "KNN": {
        'n_neighbors': [15]
    },
    "MLP": {
          'hidden_layer_sizes': [(32, 16)],
          'activation': ['relu']
    },
    "BaggingClassifier": {
        'n_estimators': [200]
    },
    "StackingClassifier": {
        'passthrough': [True]
    }
}

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


In [ ]:
# Define custom scorers globally, as they can be reused across different evaluations.
specificity_scorer = make_scorer(recall_score, pos_label=0) # Specificity is recall for the negative class
g_mean_scorer = make_scorer(geometric_mean_score)

# Define the full set of scoring metrics to be used in cross_validate
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'mcc': make_scorer(matthews_corrcoef),
    'kappa': make_scorer(cohen_kappa_score),
    'specificity': specificity_scorer,
    'g_mean': g_mean_scorer
}


In [ ]:
def evaluate_models(X, y, label):
    print(f"\n===== {label} ====")

    for name, model in models.items():
        print(f"\nModel: {name}")
        random_search = RandomizedSearchCV(
                    model,
                    param_distributions=params.get(name),
                    n_iter=5 if name!="Stacking Classifier" else 1,
                    cv=skf,
                    scoring=scoring,
                    random_state=42,
                    refit='roc_auc',
                    n_jobs=-1
                )
        random_search.fit(X, y)
        print("Best Parameters:", random_search.best_params_)
        current_model_for_cv = random_search.best_estimator_ # Use the best estimator found by RandomizedSearchCV



        scores = cross_validate(current_model_for_cv, X, y, cv=skf, scoring=scoring, n_jobs=-1)

        metrics = {
            "label": label,
            "Model": name,
            "Accuracy":  f"{np.mean(scores['test_accuracy']):.4f}",
            "Precision": f"{np.mean(scores['test_precision']):.4f}",
            "Recall":    f"{np.mean(scores['test_recall']):.4f}",
            "F1 Score":  f"{np.mean(scores['test_f1']):.4f}",
            "ROC-AUC":   f"{np.mean(scores['test_roc_auc']):.4f}",
            "MCC":       f"{np.mean(scores['test_mcc']):.4f}",
            "Kappa":     f"{np.mean(scores['test_kappa']):.4f}",
            "Specificity": f"{np.mean(scores['test_specificity']):.4f}",
            "G-Mean": f"{np.mean(scores['test_g_mean']):.4f}"
        }
        results.append(metrics)
        # Print the result
        print(f"Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall:    {np.mean(scores['test_recall']):.4f}")
        print(f"F1 Score:  {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC:   {np.mean(scores['test_roc_auc']):.4f}")
        print(f"MCC:       {np.mean(scores['test_mcc']):.4f}")
        print(f"Kappa:     {np.mean(scores['test_kappa']):.4f}")
        print(f"Specificity: {np.mean(scores['test_specificity']):.4f}")
        print(f"G-Mean: {np.mean(scores['test_g_mean']):.4f}")

In [ ]:
results = [] # Initialize results before calling evaluate_models
evaluate_models(X_train_sm, y_train_sm, "ADASYN")


===== ADASYN ====

Model: Logistic Regression


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=5. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Parameters: {'C': 1.0}
Accuracy:  0.6892
Precision: 0.7397
Recall:    0.5829
F1 Score:  0.6520
ROC-AUC:   0.7466
MCC:       0.3870
Kappa:     0.3782
Specificity: 0.7952
G-Mean: 0.6808

Model: Decision Tree


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=5. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Parameters: {'min_samples_split': 40, 'min_samples_leaf': 2, 'max_depth': 20}
Accuracy:  0.9015
Precision: 0.9298
Recall:    0.8684
F1 Score:  0.8980
ROC-AUC:   0.9533
MCC:       0.8047
Kappa:     0.8030
Specificity: 0.9345
G-Mean: 0.9008

Model: Random Forest


...

Best Parameters: {'n_estimators': 200}
Accuracy:  0.9416
Precision: 0.9585
Recall:    0.9231
F1 Score:  0.9404
ROC-AUC:   0.9793
MCC:       0.8838
Kappa:     0.8832
Specificity: 0.9601
G-Mean: 0.9414

Model: StackingClassifier


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=5. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Parameters: {'passthrough': True}


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Accuracy:  0.9408
Precision: 0.9443
Recall:    0.9367
F1 Score:  0.9405
ROC-AUC:   0.9830
MCC:       0.8816
Kappa:     0.8815
Specificity: 0.9448
G-Mean: 0.9407


In [19]:
results_df = pd.DataFrame(results)

In [24]:
from google.colab import drive
results_df.to_excel('/content/adasyn.xlsx', index=False)